# U-Net (ResNet34 backbone) Road Segmentation — Google Colab edition

Trains U-Net + ResNet34 on the [DeepGlobe Road Extraction dataset](https://www.kaggle.com/datasets/balraj98/deepglobe-road-extraction-dataset) on a **free Google Colab GPU** runtime (T4, 16 GB VRAM).

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

This version accounts for a few things a plain Kaggle-notebook version wouldn't need to worry about:

- There is no `/kaggle/input` on Colab, so the dataset is downloaded via `kagglehub` and reorganized into the `Train/Validation/Test` split this notebook expects — see the *Dataset* section below.
- `/content` is wiped whenever the runtime recycles, so the prepared dataset and checkpoints/logs are cached on Google Drive instead — later sessions reuse them instead of re-downloading and reprocessing ~2GB of imagery, and training auto-resumes from the last checkpoint if the session gets cut off.
- Free sessions can be killed after roughly 12h, or earlier if idle — rerunning the training cell after a disconnect will pick up where it left off.
- Only ~2 vCPUs are available, so `DataLoader` worker count is kept low.
- Mixed precision (`16-mixed`) is enabled to make better use of the T4's Tensor Cores.

In [ ]:
!nvidia-smi

In [ ]:
from IPython.display import clear_output
import importlib

%pip install -q -U segmentation-models-pytorch pytorch-lightning
if importlib.util.find_spec("cv2") is None:
    %pip install -q opencv-python-headless

clear_output()

In [ ]:
import os
import cv2
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.optim import lr_scheduler
import segmentation_models_pytorch as smp
import pytorch_lightning as pl
import torchvision.transforms

## Dataset

Uses the [DeepGlobe Road Extraction dataset](https://www.kaggle.com/datasets/balraj98/deepglobe-road-extraction-dataset) (`balraj98/deepglobe-road-extraction-dataset`). Only its `train/` split ships ground-truth masks — `valid/`/`test/` are the original Kaggle competition's unlabeled holdouts — so the cells below download the raw data via `kagglehub` and carve our own `Train`/`Validation`/`Test` split out of the labeled `train/` pairs (`<id>_sat.jpg` / `<id>_mask.png`), matching them into `image/`/`label/` folders that `TGRSDataset` expects.

The prepared split is cached under Google Drive (`PREPARED_DATASET_ROOT`), so it's only downloaded and rebuilt once — later sessions reuse it directly, which matters given Colab's per-session and weekly GPU-hour limits.

The first download will prompt you to authenticate with Kaggle (upload a `kaggle.json` API token, or set the `KAGGLE_USERNAME`/`KAGGLE_KEY` Colab secrets beforehand).

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = '/content/drive/MyDrive/deepglobe-road-unet/checkpoints'
    PREPARED_DATASET_ROOT = '/content/drive/MyDrive/deepglobe-road-unet/prepared-dataset'
    RAW_DATASET_ROOT = None  # downloaded via kagglehub in the next cell
elif os.path.isdir('/kaggle/input'):
    # running on a Kaggle notebook with the dataset attached as input
    CHECKPOINT_DIR = './checkpoints'
    PREPARED_DATASET_ROOT = '/kaggle/working/prepared-dataset'
    RAW_DATASET_ROOT = '/kaggle/input/deepglobe-road-extraction-dataset'
else:
    CHECKPOINT_DIR = './checkpoints'
    PREPARED_DATASET_ROOT = './prepared-dataset'
    RAW_DATASET_ROOT = None  # downloaded via kagglehub in the next cell

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

### Running on Kaggle

This notebook also runs unmodified on a Kaggle notebook:

- Attach the `balraj98/deepglobe-road-extraction-dataset` dataset as a notebook input.
- Under notebook *Settings*, turn on **Internet** (needed for the `pip install` cells) and pick a GPU accelerator.
- Picking **GPU T4 x2** gives two GPUs. The trainer cell below detects this (`NUM_GPUS = torch.cuda.device_count()`) and automatically switches to PyTorch Lightning's `ddp_notebook` strategy, which splits each epoch's batches across both GPUs (plain `ddp` doesn't work from inside a notebook process).
- Unlike the Colab/Drive setup above, `/kaggle/working` isn't guaranteed to survive between separate interactive sessions. If you need multi-session resume, commit the notebook (**Save Version**) so the checkpoints are kept as output, then attach that version's output as an input on the next run and copy `last.ckpt` back into `CHECKPOINT_DIR` before training.

In [ ]:
import random
import shutil
from pathlib import Path

SAT_SUFFIX = "_sat.jpg"
MASK_SUFFIX = "_mask.png"


def _link_or_copy(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        return
    try:
        dst.symlink_to(src.resolve())
    except OSError:
        shutil.copy2(src, dst)


def prepare_deepglobe_dataset(raw_root, out_root, val_frac=0.1, test_frac=0.1, seed=42):
    """Split DeepGlobe's labeled train/ pairs into Train/Validation/Test/{image,label}/."""
    train_dir = Path(raw_root) / "train"
    pairs = []
    for sat_path in sorted(train_dir.glob(f"*{SAT_SUFFIX}")):
        img_id = sat_path.name[: -len(SAT_SUFFIX)]
        mask_path = train_dir / f"{img_id}{MASK_SUFFIX}"
        if mask_path.exists():
            pairs.append((img_id, sat_path, mask_path))

    rng = random.Random(seed)
    rng.shuffle(pairs)
    n_test = int(len(pairs) * test_frac)
    n_val = int(len(pairs) * val_frac)
    splits = {
        "Test": pairs[:n_test],
        "Validation": pairs[n_test:n_test + n_val],
        "Train": pairs[n_test + n_val:],
    }

    out_root = Path(out_root)
    counts = {}
    for split_name, split_pairs in splits.items():
        image_dir = out_root / split_name / "image"
        label_dir = out_root / split_name / "label"
        for img_id, sat_path, mask_path in split_pairs:
            # Same stem in both folders (extensions may differ) so TGRSDataset can pair them.
            _link_or_copy(sat_path, image_dir / f"{img_id}{sat_path.suffix}")
            _link_or_copy(mask_path, label_dir / f"{img_id}{mask_path.suffix}")
        counts[split_name] = len(split_pairs)
    return counts


KAGGLE_DATASET_REF = "balraj98/deepglobe-road-extraction-dataset"

if not os.path.isdir(os.path.join(PREPARED_DATASET_ROOT, "Train", "image")):
    if RAW_DATASET_ROOT is None:
        %pip install -q kagglehub
        import kagglehub
        RAW_DATASET_ROOT = kagglehub.dataset_download(KAGGLE_DATASET_REF)
    stats = prepare_deepglobe_dataset(RAW_DATASET_ROOT, PREPARED_DATASET_ROOT)
    print("Prepared DeepGlobe splits:", stats)

DATASET_ROOT = PREPARED_DATASET_ROOT

In [ ]:
for split in ("Train", "Validation", "Test"):
    split_path = os.path.join(DATASET_ROOT, split)
    assert os.path.isdir(split_path), f"Missing {split_path} -- run the dataset cell above first"
print("Using dataset at:", DATASET_ROOT)

In [ ]:
class TGRSDataset(Dataset):
    def __init__(self, dataset_path,
                 image_folder_name = 'image',
                 label_folder_name = 'label',
                 image_transform = None,
                 label_transform = None):
        super().__init__()
        self.image_transform = image_transform
        self.label_transform = label_transform

        self.dataset_path = dataset_path
        self.image_folder_name = image_folder_name
        self.label_folder_name = label_folder_name

        self.image_folder_path = os.path.join(dataset_path, self.image_folder_name)
        self.label_folder_path = os.path.join(dataset_path, self.label_folder_name)

        self.images, self.labels = self.load_dataset()
    def load_dataset(self) -> [list[str], list[str]]:
        image_names = sorted(os.listdir(self.image_folder_path))
        label_by_stem = {os.path.splitext(name)[0]: name for name in os.listdir(self.label_folder_path)}
        images, labels = [], []
        for name in image_names:
            stem = os.path.splitext(name)[0]
            if stem not in label_by_stem:
                continue
            images.append(os.path.join(self.image_folder_path, name))
            labels.append(os.path.join(self.label_folder_path, label_by_stem[stem]))
        return images, labels

    def __getitem__(self, idx):

        image_path, label_path = self.images[idx], self.labels[idx]
        image = cv2.imread(image_path)
        label = cv2.imread(label_path, 0)
        if self.image_transform:
            image = self.image_transform(image)
        if self.label_transform:
            label = self.label_transform(label)
        # label = label.round()
        return image, label
    def __len__(self):
        return len(self.images)

In [ ]:
def set_seed(seed: int = 42) -> None:
    # Set seed for PyTorch
    torch.manual_seed(seed)

    # Set seed for CUDA (if using GPUs)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # For multi-GPU setups

    # Ensure deterministic behavior for PyTorch operations
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
class Round():
    def __init__(self):
        pass
    def __call__(self, image):
        return image.round()

In [ ]:
NUM_GPUS = torch.cuda.device_count()
# Colab free tier only has ~2 vCPUs; Kaggle gives more, so scale worker count up there
NUM_WORKERS = min(2, os.cpu_count() or 2) if IN_COLAB else min(4, os.cpu_count() or 4)
SHUFFLE_DATASET = True
BATCH_SIZE = 32
SEED = 42
MAX_EPOCHS = 200
OUT_CLASSES = 1

IMAGE_SIZE = 224

image_transform = torchvision.transforms.Compose([torchvision.transforms.ToTensor(),
                                            torchvision.transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))])

label_transform = torchvision.transforms.Compose([torchvision.transforms.ToTensor(),
                                            torchvision.transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                                            Round()])

torch.set_float32_matmul_precision('medium')  # use Tensor Cores on the T4
set_seed(SEED)

In [ ]:
train_dataset = TGRSDataset(dataset_path = os.path.join(DATASET_ROOT, 'Train'), image_folder_name = 'image', label_folder_name = 'label', image_transform = image_transform, label_transform = label_transform)
val_dataset = TGRSDataset(dataset_path = os.path.join(DATASET_ROOT, 'Validation'), image_folder_name = 'image', label_folder_name = 'label', image_transform = image_transform, label_transform = label_transform)
test_dataset = TGRSDataset(dataset_path = os.path.join(DATASET_ROOT, 'Test'), image_folder_name = 'image', label_folder_name = 'label', image_transform = image_transform, label_transform = label_transform)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=SHUFFLE_DATASET, num_workers = NUM_WORKERS, pin_memory=True, persistent_workers = NUM_WORKERS > 0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers = NUM_WORKERS, pin_memory=True, persistent_workers = NUM_WORKERS > 0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers = NUM_WORKERS, pin_memory=True, persistent_workers = NUM_WORKERS > 0)
# Under DDP each GPU process only sees a 1/NUM_GPUS shard of train_loader (DistributedSampler),
# so the per-process step count -- and hence the cosine schedule below -- must scale down with it.
T_MAX = MAX_EPOCHS * len(train_loader) // max(NUM_GPUS, 1)

In [ ]:
class TGRSModel(pl.LightningModule):
    def __init__(self, arch,  encoder_name, t_max, in_channels, out_classes, **kwargs):
        super().__init__()
        self.model = smp.create_model(
            arch,
            encoder_name=encoder_name,
            in_channels=in_channels,
            classes=out_classes,

            **kwargs,
        )

        self.t_max = t_max

        # preprocessing parameteres for image
        params = smp.encoders.get_preprocessing_params(encoder_name)
        self.register_buffer("std", torch.tensor(params["std"]).view(1, 3, 1, 1))
        self.register_buffer("mean", torch.tensor(params["mean"]).view(1, 3, 1, 1))

        # for image segmentation dice loss could be the best first choice
        self.loss_fn = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)

        # initialize step metics
        self.training_step_outputs = []
        self.validation_step_outputs = []
        self.test_step_outputs = []

    def forward(self, image):
        # normalize image here
        image = (image - self.mean) / self.std
        mask = self.model(image)
        return mask

    def shared_step(self, batch, stage):
        image, mask = batch

        # Shape of the image should be (batch_size, num_channels, height, width)
        # if you work with grayscale images, expand channels dim to have [batch_size, 1, height, width]
        assert image.ndim == 4

        # Check that image dimensions are divisible by 32,
        # encoder and decoder connected by `skip connections` and usually encoder have 5 stages of
        # downsampling by factor 2 (2 ^ 5 = 32); e.g. if we have image with shape 65x65 we will have
        # following shapes of features in encoder and decoder: 84, 42, 21, 10, 5 -> 5, 10, 20, 40, 80
        # and we will get an error trying to concat these features
        h, w = image.shape[2:]
        assert h % 32 == 0 and w % 32 == 0
        assert mask.ndim == 4

        # Check that mask values in between 0 and 1, NOT 0 and 255 for binary segmentation
        assert mask.max() <= 1.0 and mask.min() >= 0

        logits_mask = self.forward(image)

        # Predicted mask contains logits, and loss_fn param `from_logits` is set to True
        loss = self.loss_fn(logits_mask, mask)

        # Lets compute metrics for some threshold
        # first convert mask values to probabilities, then
        # apply thresholding
        prob_mask = logits_mask.sigmoid()
        pred_mask = (prob_mask > 0.5).float()

        # We will compute IoU metric by two ways
        #   1. dataset-wise
        #   2. image-wise
        # but for now we just compute true positive, false positive, false negative and
        # true negative 'pixels' for each image and class
        # these values will be aggregated in the end of an epoch
        tp, fp, fn, tn = smp.metrics.get_stats(
            pred_mask.long(), mask.long(), mode="binary"
        )
        return {
            "loss": loss,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
        }

    def shared_epoch_end(self, outputs, stage):
        # aggregate step metics
        tp = torch.cat([x["tp"] for x in outputs])
        fp = torch.cat([x["fp"] for x in outputs])
        fn = torch.cat([x["fn"] for x in outputs])
        tn = torch.cat([x["tn"] for x in outputs])

        # per image IoU means that we first calculate IoU score for each image
        # and then compute mean over these scores
        per_image_iou = smp.metrics.iou_score(
            tp, fp, fn, tn, reduction="micro-imagewise"
        )

        # dataset IoU means that we aggregate intersection and union over whole dataset
        # and then compute IoU score. The difference between dataset_iou and per_image_iou scores
        # in this particular case will not be much, however for dataset
        # with "empty" images (images without target class) a large gap could be observed.
        # Empty images influence a lot on per_image_iou and much less on dataset_iou.
        dataset_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro")
        metrics = {
            f"{stage}_per_image_iou": per_image_iou,
            f"{stage}_dataset_iou": dataset_iou,
        }

        self.log_dict(metrics, prog_bar=True)

    def training_step(self, batch, batch_idx):
        train_loss_info = self.shared_step(batch, "train")
        # append the metics of each step to the
        self.training_step_outputs.append(train_loss_info)
        return train_loss_info

    def on_train_epoch_end(self):
        self.shared_epoch_end(self.training_step_outputs, "train")
        # empty set output list
        self.training_step_outputs.clear()
        return

    def validation_step(self, batch, batch_idx):
        valid_loss_info = self.shared_step(batch, "valid")
        self.validation_step_outputs.append(valid_loss_info)
        return valid_loss_info

    def on_validation_epoch_end(self):
        self.shared_epoch_end(self.validation_step_outputs, "valid")
        self.validation_step_outputs.clear()
        return

    def test_step(self, batch, batch_idx):
        test_loss_info = self.shared_step(batch, "test")
        self.test_step_outputs.append(test_loss_info)
        return test_loss_info

    def on_test_epoch_end(self):
        self.shared_epoch_end(self.test_step_outputs, "test")
        # empty set output list
        self.test_step_outputs.clear()
        return

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=2e-4)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.t_max, eta_min=1e-5)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",
                "frequency": 1,
            },
        }
        return

In [ ]:
model = TGRSModel("Unet", "resnet34", T_MAX, in_channels=3, out_classes=1)

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename="unet-resnet34-{epoch:03d}-{valid_dataset_iou:.3f}",
    monitor="valid_dataset_iou",
    mode="max",
    save_last=True,
)

# Free Colab sessions can be cut off mid-run; resume from the last checkpoint saved to Drive if there is one
last_ckpt_path = os.path.join(CHECKPOINT_DIR, "last.ckpt")
resume_ckpt = last_ckpt_path if os.path.isfile(last_ckpt_path) else None

# "ddp_notebook" (not plain "ddp") is required to run multi-GPU training from inside a notebook
# process -- e.g. Kaggle's "GPU T4 x2" accelerator -- since regular ddp launches worker processes
# via the command line, which doesn't work in a Colab/Kaggle/Jupyter kernel.
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    log_every_n_steps=1,
    accelerator="auto",
    devices="auto",
    strategy="ddp_notebook" if NUM_GPUS > 1 else "auto",
    precision="16-mixed",
    default_root_dir=CHECKPOINT_DIR,
    callbacks=[checkpoint_callback],
)

trainer.fit(
    model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
    ckpt_path=resume_ckpt,
)

In [ ]:
# run validation dataset
valid_metrics = trainer.validate(model, dataloaders=val_loader, verbose=False)
print(valid_metrics)

In [ ]:
# run test dataset
test_metrics = trainer.test(model, dataloaders=test_loader, verbose=False)
print(test_metrics)

In [ ]:
# NOTE: the model lives on the GPU after training, so inputs must be moved there too before calling it directly
device = next(model.parameters()).device

batch = next(iter(test_loader))
with torch.no_grad():
    model.eval()
    images, masks = batch
    logits = model(images.to(device))
pr_masks = logits.sigmoid().cpu()
images = images.cpu()
# pr_masks[pr_masks >= 0.5] = 255
# pr_masks[pr_masks < 0.5] = 0
for idx, (image, gt_mask, pr_mask) in enumerate(
    zip(images, masks, pr_masks)
):
    if idx <= 4:
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 3, 1)
        plt.imshow(image.numpy().transpose(1, 2, 0))
        plt.title("Image")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(gt_mask.numpy().squeeze())
        plt.title("Ground truth")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(pr_mask.numpy().squeeze())
        plt.title("Prediction")
        plt.axis("off")
        plt.show()
    else:
        break